In [3]:
from astropy.coordinates import SkyCoord
from astropy import units as u
import numpy as np
from astropy.table import Table, vstack
import healsparse as hsp
import healpy as hp
from hpgeom import hpgeom
import skyproj
import matplotlib.pyplot as plt

import sys
import os
#Goes up a directories to get the updated astroquery
#I really need to fix this, maybe github submodules or enforcing a version of astroquery
#I think it's version 0.4.11 ?

# Get the absolute path of the parent directory
parent_dir = os.path.abspath("../..")
# Insert it at the beginning of the path list so Python looks there first
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    #sys.path.append(parent_dir)
# Now you can import your package normally
from astroquery_upper.astroquery.esa.euclid import Euclid #need astroquery v0.4.10
from ugali.utils import healpix #.utils import healpix #need ugali but that isn't in these kernels by default


# NERSC paths
my_path =  '/global/u2/k/kexcell/'
data_path = '/global/u2/k/kexcell/ufd-dkm/euclid_data/maps'
my_plotspath = '/global/u2/k/kexcell/ufd-dkm/plots'

Euclid.get_status_messages()  # check archive isn't degraded

In [ ]:
ras = ['58', '59']
decs = ['-49', '-48']
corners_str = ''
for ra in ras:
    for dec in decs:
        corners_str += f'{ra}, {dec}, '
corners_str.removesuffix(', ')

# NOTE:
There's a __get_tile_catalogue_list method of Euclid  
Searchable products by tile_index: 'DpdMerBksMosaic', 'dpdPhzPfOutputForL3', 'dpdPhzPfOutputCatalog',
            'dpdMerFinalCatalog','dpdSpePfOutputCatalog', 'dpdSheLensMcChains', 'dpdHealpixBitMaskVMPZ',
            'dpdHealpixFootprintMaskVMPZ', 'dpdHealpixCoverageVMPZ', 'dpdHealpixDepthMapVMPZ','dpdHealpixInfoMapVMPZ',
            'dpdSheBiasParams', 'dpdSheLensMcFinalCatalog', 'dpdSheLensMcRawCatalog', 'dpdSheMetaCalFinalCatalog',
            'dpdSheMetaCalRawCatalog', 'dpdSleDetectionOutput', 'dpdSleModelOutput', 'DpdSirCombinedSpectra',
            'DpdMerSegmentationMap'

Loading in all the possible tables from q1...

In [ ]:
tables = Euclid.load_tables(only_names=True, include_shared_tables=True)
tile = [t.name for t in tables if 'q1' in t.name.lower()]
tile

Loading in all the tables from q1 with vmpz (maps), outputs these possibilities:  
q1.vmpz_healpix_bitmask - no clue what this is  
q1.vmpz_healpix_coverage - I think this may be what I want  
q1.vmpz_healpix_depthmap - gives magnitude limits, can be useful  
q1.vmpz_healpix_footprint_mask - may also be what I want...? might be more general. more to play with  
q1.vmpz_healpix_infomap - no clue what this is  
q1.mer_segmentation_map - no clue what this is  

In [ ]:
map_names = [t.name for t in tables if 'q1.vmpz' in t.name.lower()]

### Here is where code actually starts

Query for the coverage map and the footprint_mask, to understand the difference better

#### combined map file

The 8 maps are different filters, so I've made the combined coverage map for VIS

In [ ]:
#ONLY RUN THIS AGAIN IF CHANGING THE FILES
'''
tiles = [tile_list[0] for tile_list in np.unique(covmap['tile_index_list'])]

file_list = []
for tile in tiles:
    i = np.where((covmap['tile_index_list']==[tile])&(covmap['filter_name']=='VIS'))[0][0]
    path = Euclid.get_product(file_name=covmap['file_name_list'][i], product_id = covmap['product_id'][i], 
                                  output_file= data_path+ f"/coverage_{tile}_vis.fits", verbose=False)
    file_list.append(data_path+ f"/coverage_{tile}_vis.fits")
map_list = []
for tile in tiles:
    map_list.append(hsp.HealSparseMap.read(data_path+ f"/coverage_{tile}_vis.fits", nside_coverage=32))
combined_map = hsp.operations.sum_union(map_list)
combined_map.write(data_path+ f"/coverage_vis_combined.fits", clobber=False)
'''

In [ ]:
coverage_vis = hsp.HealSparseMap.read(data_path+ f"/coverage_vis_combined.fits")

In [ ]:
## Loading in a random ra/dec circle of MER catalog data. Commented out sections are rejected ways of querying (for now)

def mer_query(ra, dec, radius=0.5):

    ## below loads in file paths to download similar to the maps
    #catalog_name = 'q1.mer_final_catalogue'
    #mer_query = f''' SELECT * FROM {catalog_name}
    #            '''
    #mer_results = Euclid.launch_job_async(mer_query).get_results()
    query = f'''
            SELECT right_ascension, declination, point_like_prob, point_like_flag,
            ellipticity, mumax_minus_mag, flux_vis_psf, fluxerr_vis_psf, spurious_flag,
            det_quality_flag, fwhm, segmentation_map_id
            FROM mer_catalogue WHERE DISTANCE({ra}, {dec}, right_ascension, declination) < {radius}
            '''
    #mer_results = Euclid.cone_search(coordinate=coord, radius=1 * u.degree).get_results()#, columns = ['tileId'])
    #or WHERE right_ascension BETWEEN ra1 AND ra2 and declination BETWEEN dec1 AND dec2
    return Euclid.launch_job_async(query, verbose=False).get_results()

mer_results = mer_query(58, -51)
mask = (mer_results['spurious_flag']==0)
#mask &= (mer_results['segmentation_map_id'] // 10**6 == tile_id)
mer_results_clean = mer_results[mask]

In [ ]:
fracdet = coverage_vis.fracdet_map(4096)

In [ ]:
#ultimately simple wants:
    #mean_fracdet = np.mean(fracdet[subpix_region_array[cut]])
#subpix_region_array seems to be the area of the sky we want to look at

#so I think I just need to figure out how to extract just the region I want from fracdet

subpix_region_array = []
nside = 32
nside_fracdet = fracdet.nside_sparse
for pix in np.unique(healpix.angToPix(nside, mer_results_clean['right_ascension'], mer_results_clean['declination'])):
    subpix_region_array.append(healpix.subpixel(pix, nside, nside_fracdet))
subpix_region_array = np.concatenate(subpix_region_array)
#map_mask = hsp.HealSparseMap.make_empty(32, 4096, np.float64)
m = fracdet[subpix_region_array]

In [ ]:
## ok this is my best guess for now
# do I need to do it for a smaller area though?
values = fracdet.get_values_pos(mer_results_clean['right_ascension'], mer_results_clean['declination'])
np.mean(values)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sp = skyproj.MollweideSkyproj(ax=ax)
sp.draw_hspmap(coverage_vis)
#if catalog is not None:
#    sp.ax.scatter(catalog['right_ascension'],catalog['declination'],s=.005,c='r')
plt.title(f'Euclid Coverage VIS Map', pad = 20)
plt.colorbar()
#plt.tight_layout()
#plt.savefig(my_plotspath + f'/property_maps/tile{tileid}_sumunion_euclidcoveragemap.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sp = skyproj.MollweideSkyproj(ax=ax)
sp.draw_hspmap(fracdet)
#if catalog is not None:
#    sp.ax.scatter(catalog['right_ascension'],catalog['declination'],s=.005,c='r')
plt.title(f'Euclid FracDet Map', pad = 20)
plt.colorbar()
#plt.tight_layout()
#plt.savefig(my_plotspath + f'/property_maps/tile{tileid}_sumunion_euclidcoveragemap.png')
plt.show()

In [ ]:
ra, dec = mer_results_clean['right_ascension'], mer_results_clean['declination']
values = fracdet.get_values_pos(ra,dec)

fig = plt.figure(figsize=(8, 5))
plt.scatter(ra, dec, c=values,s=0.1)
plt.xlabel("Right Ascension (deg)")
plt.ylabel("Declination (deg)")
plt.colorbar(label="FracDet")
plt.show()

#### If you want to just query one tile at a time

In [ ]:
def map_query(map_name):
    query = f"SELECT * FROM {map_name}"
    results = Euclid.launch_job_async(query).get_results()
    results.sort('file_path')
    return results

covmap = map_query('q1.vmpz_healpix_coverage')
footprint = map_query('q1.vmpz_healpix_footprint_mask')

def map_tile_queryNsave(tile_id, map_name, map_results):
    i_list = np.where((covmap['tile_index_list']==[tile_id])&(covmap['filter_name']=='VIS'))[0]
    n=1
    map_list = []
    for i in i_list:
        output_path = data_path+ f"/{map_name}_{tile_id}_{n}.fits"
        path = Euclid.get_product(file_name=map_results['file_name_list'][i], product_id = map_results['product_id'][i], 
                                  output_file=output_path, verbose=False)
        # ! need to figure out if this nside_coverage is telling hsp which coverage to read in 
        # or if it makes an assumption about the properties of this map
        m = hp.read_map(output_path)
        nside = hp.get_nside(m)
        map_list.append(hsp.HealSparseMap.read(output_path, nside_coverage=32))
        n+=1
    return map_list

tile_id = 102018666 #102018212
coverage_singlemaps = map_tile_queryNsave(tile_id, 'coverage', covmap)
footprint_singlemaps = map_tile_queryNsave(tile_id, 'footprint', footprint)

In [ ]:
print(len(coverage_singlemaps))

In [ ]:
vpix, ra, dec = coverage_singlemaps[0].valid_pixels_pos(return_pixels=True)
ra_centerish = np.median(ra)
dec_centerish = np.median(dec)

#### Need to decide how to best combine these maps...

In [ ]:
coverage_combinedmap = hsp.operations.sum_union(coverage_singlemaps)
footprint_combinedmap = hsp.operations.sum_union(footprint_singlemaps)

In [ ]:
def plot_maps(singlemaps, combinedmap, catalog=None):
    fig, ax = plt.subplots(3,3,figsize=(14, 10))
    for i in range(len(singlemaps)):
        sp = skyproj.MollweideSkyproj(ax=ax.flatten()[i])
        sp.draw_hspmap(singlemaps[i])
        sp.ax.set_xlabel("Right Ascension", fontsize = 7)
        sp.ax.set_ylabel("Declination", fontsize = 7)
        sp.ax.tick_params(axis = "x", labelsize=7)
        sp.ax.tick_params(axis = "y", labelsize=7)
    plt.suptitle(f'Euclid Single Maps, Tile {tile_id}', y=.9)
    plt.colorbar(ax = ax[2][2])
    #plt.tight_layout()
    #plt.savefig(my_plotspath + f'/property_maps/tile{tileid}_single_euclidcoveragemap.png')
    plt.show()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sp = skyproj.MollweideSkyproj(ax=ax)
    sp.draw_hspmap(combinedmap,cmap='Greys')
    if catalog is not None:
        sp.ax.scatter(catalog['right_ascension'],catalog['declination'],s=.005,c='r')
    plt.title(f'Euclid Union Sum, Tile {tile_id}', pad = 20)
    plt.colorbar()
    #plt.tight_layout()
    #plt.savefig(my_plotspath + f'/property_maps/tile{tileid}_sumunion_euclidcoveragemap.png')
    plt.show()

In [ ]:
plot_maps(coverage_singlemaps, coverage_combinedmap, mer_results_clean)

In [ ]:
ra_col, dec_col = mer_results['right_ascension'], mer_results['declination']
nside = 512
pixels = healpix.ang2pix(nside, ra_col, dec_col, nest=True)
sorted_pixels = np.unique(np.sort(pixels))
labels = []
for pix in sorted_pixels:
    ra_pix, dec_pix = healpix.pix2ang(nside, pix, nest = True)
    labels.append(f'({round(ra_pix,3)}, {round(dec_pix,3)})')

fig, ax1 = plt.subplots()
ax1.hist(pixels, bins=50)
ax1.set(xlabel='pixels', ylabel='number', yscale='log')
second = ax1.secondary_xaxis(location=1)
second.set_xticks(sorted_pixels, labels=labels,rotation=90)
plt.tight_layout()
plt.savefig(my_plotspath + f'/pixels_histogram.png')
plt.show()

In [ ]:
labels

In [ ]:
plot_maps(footprint_singlemaps, footprint_combinedmap, mer_results_clean)

In [ ]:
fracdet = combined_map.fracdet_map(512)
#fracdet_zero = np.tile(0,len(fracdet))
#cut = (fracdet != hp.UNSEEN)
#fracdet_zero[cut] = fracdet[cut]

In [ ]:
## This is something Peter used for querying, he wanted images though

#query = f"SELECT tile_index, ra, dec FROM q1.mosaic_product"
#query = "SELECT * FROM q1.observation_mosaic"
#results = Euclid.launch_job_async(query).get_results()
#results.sort('tile_index') #[['tile_index','ra','dec']]
#results

In [ ]:
## this cell just checks that there's never more than one tile index for a row
for row in range(len(map_results['tile_index_list'])):
    if len(map_results['tile_index_list'][row]) < 1:
        print(map_results[['file_path','tile_index_list']][row])

In [ ]:
start = 8*5
n = 1
for i in range(start, start+8):
    tileid = results['tile_index_list'][i][0]
    print(tileid)
    output_path = f"euclid_data/coverage_map_{tileid}_{n}.fits"
    print(output_path)
    path = Euclid.get_product(file_name=results['file_name_list'][i], product_id = results['product_id'][i], 
                              output_file=output_path, verbose=False)
    n += 1

fig, ax = plt.subplots(3,3,figsize=(14, 10))
covmap_list = []
for n in range(1,9):
    # ! need to figure out if this nside_coverage is telling hsp which coverage to read in 
        # or if it makes an assumption about the properties of this map
    mapa = hsp.HealSparseMap.read(f'euclid_data/coverage_map_{tileid}_{n}.fits', nside_coverage=32)
    covmap_list.append(mapa)
    sp = skyproj.MollweideSkyproj(ax=ax.flatten()[n-1])
    sp.draw_hspmap(mapa)
    sp.ax.set_xlabel("Right Ascension", fontsize = 7)
    sp.ax.set_ylabel("Declination", fontsize = 7)
    sp.ax.tick_params(axis = "x", labelsize=7)
    sp.ax.tick_params(axis = "y", labelsize=7)
plt.suptitle(f'Euclid Coverage Single Maps, Tile {tileid}', y=.9)
plt.colorbar(ax = ax[2][2])
#plt.tight_layout()
plt.savefig(my_plotspath + f'/property_maps/tile{tileid}_single_euclidcoveragemap.png')
plt.show()

covmap = hsp.operations.sum_union(covmap_list)


In [ ]:
'''
valid = Euclid.get_valid_le3_configuration_values()
for row in range(len(valid)):
    if 'VMPZ' in valid['product_type'][row]:
    #if valid['product_type'][row].lower() == "dpdHealpixFootprintMaskVMPZ".lower():
        print(valid[row])


results = Euclid.get_scientific_product_list(
    category = "Internal Data Products",
    group = "GeneralMasksVMPZ",
    product_type= 'DpdHealpixEffectiveCoverageVMPZ', # or "DpdHealpixFootprintMaskVMPZ", 
    tile_index = 22,
    dataset_release="Q1_R1"
)

print(results)#.columns)
'''

'''

fig, ax = plt.subplots(figsize=(8, 5))
sp = skyproj.MollweideSkyproj(ax=ax)
sp.draw_hspmap(footprint)
plt.title('Euclid Footprint', pad = 20)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
sp = skyproj.MollweideSkyproj(ax=ax)
sp.draw_hspmap(depthmap)
plt.title('Euclid Depth Map', pad = 20)
plt.colorbar()
plt.show()
'''
#for images

'''
SELECT mosaic_product.file_name, mosaic_product.mosaic_product_oid, mosaic_product.tile_index, mosaic_product.instrument_name, mosaic_product.filter_name, mosaic_product.category, mosaic_product.second_type, mosaic_product.ra, mosaic_product.dec, mosaic_product.technique 
FROM sedm.mosaic_product 
WHERE (release_name='Q1_R1') 
AND ((instrument_name='NISP') OR (instrument_name='VIS')) 
AND (category='SCIENCE') 
AND ((mosaic_product.fov IS NOT NULL AND INTERSECTS(CIRCLE('ICRS',53.13,-28.1,1),mosaic_product.fov)=1)) 
ORDER BY mosaic_product.tile_index ASC
'''
#wget -O 'EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE102044185-15D02F_20241021T005729.282854Z_00.00.fits' 'https://eas.esac.esa.int/sas-dd/data?file_name=EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE102044185-15D02F_20241021T005729.282854Z_00.00.fits&release=sedm&RETRIEVAL_TYPE=FILE'